In [1]:
import fastf1
import pandas as pd
import gc, time

fastf1.Cache.enable_cache('./cache')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

all_laps = []

# loop over every race, every season from 2021 to 2025
for i in range(2021, 2026):
    circuit = fastf1.get_event_schedule(i, include_testing=False)
    for index, event in circuit.iterrows():
        circuit_name = event['Location']
        print(f"Year: {i} | Valid Circuit String: {circuit_name}")
        session = fastf1.get_session(i, circuit_name, 'R')
        session.load()
        time.sleep(2)  # small delay so we don't spam FastF1's servers with requests and avoid rate limit
        laps = session.laps

        # keep only green flag laps with no pit 
        clean_laps = laps[
            laps['PitInTime'].isna() &
            laps['PitOutTime'].isna() &
            (laps['TrackStatus'] == '1')
        ]
        if clean_laps.empty:
            continue
        clean_laps = clean_laps.copy()
        clean_laps['StintLength'] = clean_laps.groupby(['Driver', 'Stint'])['LapNumber'].transform('count')

        # gap to the car directly ahead, by track position
        def gap_to_ahead(group):
            group = group.sort_values('Position')
            group['GapToCarAhead'] = group['Time'] - group['Time'].shift(1)
            return group

        clean_laps = clean_laps.groupby('LapNumber', group_keys=False).apply(gap_to_ahead)

        # gap to the lead racer
        def gap_to_leader(group):
            leader_time = group.loc[group['Position'] == 1, 'Time'].values
            if len(leader_time) > 0:
                group['GapToLeader'] = group['Time'] - leader_time[0]
            else:
                group['GapToLeader'] = pd.NaT
            return group

        clean_laps = clean_laps.groupby('LapNumber', group_keys=False).apply(gap_to_leader)
        clean_laps['LapTime'] = clean_laps['LapTime'].dt.total_seconds()
        clean_laps['GapToLeader'] = pd.to_timedelta(clean_laps['GapToLeader']).dt.total_seconds()
        clean_laps['GapToCarAhead'] = clean_laps['GapToCarAhead'].dt.total_seconds()
        clean_laps[['GapToLeader', 'GapToCarAhead']] = clean_laps[['GapToLeader', 'GapToCarAhead']].fillna(0)

        final = clean_laps[['Driver', 'LapNumber', 'LapTime', 'TyreLife', 'Compound', 'Position',
                             'GapToLeader', 'GapToCarAhead', 'StintLength']]
        final['Season'] = i
        final['Circuit'] = circuit_name
        final = final[['Season', 'Circuit', 'Driver', 'LapNumber', 'LapTime', 'TyreLife', 'Compound', 'Position',
                        'GapToLeader', 'GapToCarAhead', 'StintLength']]
        all_laps.append(final)

        if 'session' in locals():
            del session
            gc.collect()

# Final saaving to master file
superset = pd.concat(all_laps, ignore_index=True)
superset.to_csv('master_dataset.csv', index=False)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Year: 2021 | Valid Circuit String: Sakhir


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '11'
core        WARNING 	Driver 44 completed the race distance 00:00.067000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '77', '4', '11', '16', '3', '55', '22', '18', '7', '99', '31', '63', '5', '47', '10', '6', '14', '9']
C:\Users\bhava\AppData\Local\Temp\ipykernel_9476\3156056791.

Year: 2021 | Valid Circuit String: Imola


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 33 completed the race distance 00:01.003000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['33', '44', '4', '16', '55', '3', '10', '18', '31', '14', '11', '22', '7', '99', '5', '47', '9', '77', '63', '6']
C:\Users\bhava\AppData\Local\Temp\ipykernel_9476\3156056791.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping column

...

Year: 2021 | Valid Circuit String: Lusail


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 44 completed the race distance 00:00.037000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '14', '11', '31', '18', '55', '16', '4', '5', '10', '3', '22', '7', '99', '47', '63', '9', '6', '77']
C:\Users\bhava\AppData\Local\Temp\ipykernel_9476\3156056791.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping column

Year: 2021 | Valid Circuit String: Jeddah


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 44 completed the race distance 00:00.180000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

df = pd.read_csv('master_dataset.csv')
pre_reg = df[df['Season'] == 2021]
post_reg = df[df['Season'] == 2022]
pre_reg = pre_reg.dropna(subset=['LapTime'])
post_reg = post_reg.dropna(subset=['LapTime'])

X_pre = pre_reg[['TyreLife', 'LapNumber', 'StintLength', 'GapToCarAhead', 'GapToLeader']]
y_pre = pre_reg['LapTime']
X_post = post_reg[['TyreLife', 'LapNumber', 'StintLength', 'GapToCarAhead', 'GapToLeader']]
y_post = post_reg['LapTime']

X_pre_train, X_pre_test, y_pre_train, y_pre_test = train_test_split(X_pre, y_pre, test_size=0.2, random_state=42)
X_post_train, X_post_test, y_post_train, y_post_test = train_test_split(X_post, y_post, test_size=0.2, random_state=42)

# fit separate Random Forests for 2021 and 2022 to compare feature importance across the regulation change
h = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_pre_train, y_pre_train)
g = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_post_train, y_post_train)

print("Pre-2022:", dict(zip(X_pre.columns, h.feature_importances_)))
print("Post-2022:", dict(zip(X_post.columns, g.feature_importances_)))

features = X_pre.columns
x = np.arange(len(features))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, h.feature_importances_, width, label='Pre-2022 (2021)')
ax.bar(x + width/2, g.feature_importances_, width, label='Post-2022 (2022)')
ax.set_xticks(x)
ax.set_xticklabels(features)
ax.set_ylabel('Feature Importance')
ax.set_title('Dirty Air Effect: Pre vs Post 2022 Regulation Change')
ax.legend()
plt.tight_layout()
plt.savefig('era_comparison.png')
plt.show()

In [9]:
# residuals from the pre 2022 model, used for the diagnostic checks 
pre_residuals = y_pre_test - h.predict(X_pre_test)
print(pre_residuals.describe())

count    4073.000000
mean        0.046633
std         4.575422
min       -28.779290
25%        -1.745200
50%        -0.125790
75%         1.663270
max        35.253680
Name: LapTime, dtype: float64


In [10]:
pre_analysis = X_pre_test.copy()
pre_analysis['Residual'] = pre_residuals
pre_analysis['Circuit'] = pre_reg.loc[X_pre_test.index, 'Circuit']

# check if any circuit has unusually high residual variance
print(pre_analysis.groupby('Circuit')['Residual'].agg(['mean', 'std']).sort_values('std', ascending=False))

                  mean       std
Circuit                         
Baku          6.689568  6.621670
São Paulo    -4.803396  5.171894
Jeddah       -0.201192  4.760127
Sochi         4.476614  4.547720
Lusail       -0.994594  4.417945
Le Castellet  4.610187  3.948429
Zandvoort    -3.114909  3.849150
Istanbul      2.640516  3.681461
Budapest     -0.384262  3.578517
Mexico City  -0.756405  3.559328
Yas Island   -0.546059  3.326516
Silverstone   1.269046  3.218383
Barcelona    -1.649006  3.207191
Austin        4.382907  3.125834
Imola        -0.599338  3.104716
Spielberg    -1.731685  3.067629
Portimão     -0.331215  2.825419
Monte Carlo  -1.347194  2.693724
Sakhir        1.212739  2.509332
Monza        -0.414364  2.114055


In [11]:
pre_analysis['Compound'] = pre_reg.loc[X_pre_test.index, 'Compound']

# check if any tyre compound shows a systematic bias
print(pre_analysis.groupby('Compound')['Residual'].agg(['mean', 'std']).sort_values('std', ascending=False))

                  mean       std
Compound                        
SOFT         -1.181787  5.321782
MEDIUM       -0.437965  4.492813
HARD          0.475917  4.330624
INTERMEDIATE  2.040516  4.021463
WET          -1.578880       NaN


In [12]:
pre_analysis['Driver'] = pre_reg.loc[X_pre_test.index, 'Driver']

# check for driver level bias in residuals
print(pre_analysis.groupby('Driver')['Residual'].agg(['mean', 'std']).sort_values('mean', ascending=False).head(10))

            mean       std
Driver                    
NOR     0.976893  4.728049
OCO     0.737818  4.278468
ALO     0.495310  5.321047
BOT     0.435348  4.237078
VET     0.382258  4.811617
STR     0.348873  5.144083
SAI     0.294247  3.404194
TSU     0.206721  5.143525
GIO     0.174576  5.658617
MAZ     0.094070  4.867444
